# Data viewer

Reads the bundled Clarius raw data and shows each frame as recorded, in lines and samples, next to the same frame scan converted to millimetres.

The samples in [`data`](../../data) are a wire phantom imaged with a linear array (L7HD3, `linear`) and a convex array (C3HD3, `convex`). Each `.raw` has a `.yml` beside it that describes the acquisition, including the probe geometry that scan conversion needs.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

libPath = "../../common/python"
if libPath not in sys.path:
    sys.path.append(libPath)
import rdataread as rd
import scanconvert as sc

DATA = "../../data/"

## Reading RF data

`read_rf` returns the beamformed RF as `lines × samples × frames`. Envelope detection and log compression give a B image, which `scanconvert` then places in millimetres from the probe geometry in the `.yml`.

In [ ]:
hdr, timestamps, rf = rd.read_rf(DATA + "convex/convex_rf.raw")
meta = sc.read_meta(DATA + "convex/convex_rf.yml")
frame = rf[:, :, 0]

b = sc.bmode(meta["type"], frame)
convert = sc.converter(meta, hdr["samples"])
print(convert.description)

limit = np.percentile(np.abs(frame), 99)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(frame.T, cmap="seismic", aspect="auto", vmin=-limit, vmax=limit); axes[0].set_title("RF")
axes[1].imshow(b.T, cmap="gray", aspect="auto"); axes[1].set_title("B, pre-scan")
axes[2].imshow(convert(b), cmap="gray", extent=convert.extent_mm); axes[2].set_title("B, scan converted (mm)")
plt.show()

## Reading IQ data

`read_iq` returns I and Q interleaved along the samples (`I, Q, I, Q, …`). Their magnitude, log compressed, gives the B image.

In [ ]:
hdr, timestamps, iq = rd.read_iq(DATA + "convex/convex_iq.raw")
meta = sc.read_meta(DATA + "convex/convex_iq.yml")
i, q = iq[:, 0::2, 0], iq[:, 1::2, 0]

b = sc.bmode(meta["type"], iq[:, :, 0])
convert = sc.converter(meta, hdr["samples"])

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(i.T, cmap="viridis", aspect="auto", vmin=-100, vmax=100); axes[0].set_title("I")
axes[1].imshow(q.T, cmap="viridis", aspect="auto", vmin=-100, vmax=100); axes[1].set_title("Q")
axes[2].imshow(b.T, cmap="gray", aspect="auto"); axes[2].set_title("B, pre-scan")
axes[3].imshow(convert(b), cmap="gray", extent=convert.extent_mm); axes[3].set_title("B, scan converted (mm)")
plt.show()

## Reading envelope data

The envelope is the scanner's own B image before scan conversion: 8-bit, already log compressed.

In [ ]:
for name in ("linear", "convex"):
    hdr, timestamps, env = rd.read_env(DATA + f"{name}/{name}_env.raw")
    meta = sc.read_meta(DATA + f"{name}/{name}_env.yml")
    convert = sc.converter(meta, hdr["samples"])

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    fig.suptitle(f"{name}: {convert.description}")
    axes[0].imshow(env[:, :, 0].T, cmap="gray", aspect="auto", vmin=0, vmax=255); axes[0].set_title("pre-scan")
    axes[1].imshow(convert(env[:, :, 0]), cmap="gray", vmin=0, vmax=255, extent=convert.extent_mm)
    axes[1].set_title("scan converted (mm)")
    plt.show()